# 24. HGB統合と時間足の検査・診断
出典: FX (2).ipynb、セルindex [51, 52, 53]。保存出力は results/imported_20260909/ を参照。
研究履歴です。実行順・Notebook内変数・元の価格CSVに依存し、エラーが出たコードも保存しています。
自動判定の文言は元実験の判定であり、監査済みの結論ではありません。全セル一括実行は再現手順ではありません。
[USER_HOME] は匿名化した元のパスです。元Notebook内の案内や依頼文は研究資料として保持しています。


## 元セルindex 51


In [ ]:
# -*- coding: utf-8 -*-
"""
USD/JPY 15分足 — HGB Champion Lock-in / Full Pipeline 検証

目的
----
Feature Engineeringへ進む前に、HistGradientBoosting (HGB) を
Random Forest (RF) と同一条件で比較し、暫定Championとして固定できるか検証する。

重要な固定ルール
--------------
1) 特徴量は現行30個のまま。ここでは増やさない。
2) 15分足、30分 horizon:
       signal = t
       entry  = Open(t+1)
       exit   = Close(t+2)
3) Validationでのみ選択:
       Calibration -> Threshold / Session -> Position Sizing
4) Testを選択に使わない。
5) 30分固定Exit。TP/SLは今回入れない。
6) Risk Engineは NO_RISK 固定。
7) round-trip cost = 0.00004 を基準とし、1.5x / 2.0xも事後ストレス。
8) 2020-2025をDevelopment OOS、2026をConfirmationとして扱う。
   2026は既に結果を見ているため「完全未使用Holdout」とは呼ばない。

Jupyterで最も簡単な実行
--------------------
    %run hgb_champion_lockin.py

このファイルは、
  A) Notebook内の15分足OHLC DataFrameを自動検出
  B) 見つからなければ現在フォルダ以下の *15m*.csv / *15min*.csv を自動探索
します。

明示する場合:
    from hgb_champion_lockin import run_from_dataframe
    result = run_from_dataframe(your_15m_dataframe)

CSV:
    python hgb_champion_lockin.py --csv "path/to/usdjpy_15m.csv"
"""

from __future__ import annotations

import argparse
import json
import math
import warnings
from dataclasses import dataclass, asdict
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, brier_score_loss


# ============================================================
# 0. 固定設定
# ============================================================

FEATURES = [
    "return_1", "return_2", "return_4", "return_8", "return_16",
    "vol_4", "vol_8", "vol_16", "vol_32",
    "ma5_distance", "ma5_slope",
    "ma10_distance", "ma10_slope",
    "ma20_distance", "ma20_slope",
    "ma50_distance", "ma50_slope",
    "ma100_distance", "ma100_slope",
    "body", "upper_wick", "lower_wick", "range_pct",
    "rsi14", "atr14",
    "distance_high_16", "distance_low_16",
    "hour_sin", "hour_cos", "weekday",
]

MODEL_NAMES = ("RANDOM_FOREST", "HIST_GRADIENT_BOOSTING")

THRESHOLDS = (0.50, 0.52, 0.54, 0.55, 0.56, 0.58, 0.60, 0.62, 0.65)
SESSION_POLICIES = ("ALL", "UTC_13_24", "UTC_21_24", "EXCLUDE_08_13")
SIZING_POLICIES = ("FIXED", "GENTLE", "MODERATE", "STRONG")
CALIBRATION_METHODS = ("RAW", "PLATT", "ISOTONIC")

COST = 0.00004
COST_STRESS_MULTIPLIERS = (1.0, 1.5, 2.0)

MIN_TRAIN_YEARS = 3
MIN_TRAIN_ROWS = 5000
MIN_EVAL_ROWS = 100
MIN_VALIDATION_TRADES = 100

DEVELOPMENT_TEST_YEARS = tuple(range(2020, 2026))
CONFIRMATION_YEAR = 2026

RANDOM_SEED = 42
BOOTSTRAP_ITERATIONS = 3000
BOOTSTRAP_BLOCK_DAYS = 10


# HGBの「ロックイン用」固定設定。
# ここをTest結果に合わせて動かしてはいけない。
# 以前のModel Tournamentと完全同一パラメータがNotebookに残っているなら、
# このブロックだけをその値へ合わせてから、結果を見る前に固定すること。
HGB_CONFIG = dict(
    learning_rate=0.05,
    max_iter=250,
    max_leaf_nodes=15,
    min_samples_leaf=30,
    l2_regularization=1.0,
    early_stopping=False,
    random_state=RANDOM_SEED,
)

RF_CONFIG = dict(
    n_estimators=250,
    max_depth=8,
    min_samples_leaf=30,
    max_features="sqrt",
    class_weight="balanced",
    random_state=RANDOM_SEED,
    n_jobs=-1,
)


# ============================================================
# 1. DataFrame / CSV の正規化
# ============================================================

def _lower_ohlc_columns(frame: pd.DataFrame) -> pd.DataFrame:
    """Open/High/Low/Close または lowercase を lowercase にそろえる。"""
    mapping = {}
    for c in frame.columns:
        low = str(c).strip().lower()
        if low in {"open", "high", "low", "close"}:
            mapping[c] = low
    out = frame.rename(columns=mapping).copy()

    need = {"open", "high", "low", "close"}
    if not need.issubset(out.columns):
        raise ValueError(
            f"OHLC列が見つかりません。必要={sorted(need)}, 現在={list(frame.columns)[:20]}"
        )
    return out[["open", "high", "low", "close"]]


def _make_datetime_index(frame: pd.DataFrame) -> pd.DataFrame:
    """DatetimeIndexが無いCSVにも対応する。"""
    out = frame.copy()
    if isinstance(out.index, pd.DatetimeIndex):
        idx = out.index
    else:
        candidates = [
            c for c in out.columns
            if str(c).strip().lower() in {
                "timestamp", "time", "datetime", "date", "local time", "gmt time"
            }
        ]
        if candidates:
            time_col = candidates[0]
        else:
            # CSVの先頭列が時刻であるケースを最後に試す
            time_col = out.columns[0]

        parsed = pd.to_datetime(out[time_col], errors="coerce", utc=True)
        if parsed.notna().mean() < 0.95:
            raise ValueError("時刻列を特定できませんでした。")
        out = out.drop(columns=[time_col])
        idx = pd.DatetimeIndex(parsed)

    # tz-awareならUTCへ。naiveならDukascopy長期データの想定でUTCとして扱うが明示する。
    if idx.tz is None:
        warnings.warn(
            "Timestampがtimezone-naiveです。今回はUTCとしてlocalizeします。"
            "元CSVがUTCでない場合は、実行前に正しいtimezoneへ変換してください。"
        )
        idx = idx.tz_localize("UTC")
    else:
        idx = idx.tz_convert("UTC")

    out.index = idx
    return out


def normalize_bars(frame: pd.DataFrame) -> pd.DataFrame:
    """
    OHLCをUTCの15分足へ正規化する。
    欠測を勝手に埋めない。重複も勝手に平均しない。
    """
    if not isinstance(frame, pd.DataFrame):
        raise TypeError("入力は pandas.DataFrame が必要です。")

    out = _make_datetime_index(frame)
    out = _lower_ohlc_columns(out)
    out = out.apply(pd.to_numeric, errors="coerce").dropna()

    out = out.sort_index()

    # 完全同一timestamp重複は、同値なら1件へ。値が違えばエラー。
    if out.index.has_duplicates:
        duplicate_groups = out.loc[out.index.duplicated(keep=False)].groupby(level=0)
        bad = []
        for ts, g in duplicate_groups:
            if len(g.drop_duplicates()) > 1:
                bad.append(ts)
        if bad:
            raise ValueError(f"同一timestampに異なるOHLCがあります。例: {bad[:3]}")
        out = out[~out.index.duplicated(keep="first")]

    if out.empty:
        raise ValueError("OHLCデータが空です。")

    arr = out.to_numpy(dtype=float)
    if not np.isfinite(arr).all() or (arr <= 0).any():
        raise ValueError("OHLCに非有限値または0以下があります。")

    if (out["high"] < out[["open", "close", "low"]].max(axis=1)).any():
        raise ValueError("High < Open/Close/Low の不整合があります。")
    if (out["low"] > out[["open", "close", "high"]].min(axis=1)).any():
        raise ValueError("Low > Open/Close/High の不整合があります。")

    # 15分グリッド
    idx = out.index
    grid_bad = (
        (idx.minute % 15 != 0)
        | (idx.second != 0)
        | (idx.microsecond != 0)
    )
    if np.asarray(grid_bad).any():
        bad_examples = idx[np.asarray(grid_bad)][:5]
        raise ValueError(f"15分グリッドでないtimestampがあります: {list(bad_examples)}")

    return out


def read_csv_flexible(path: str | Path) -> pd.DataFrame:
    """一般的なDukascopy/Notebook保存CSVを読む。"""
    path = Path(path)
    raw = pd.read_csv(path)
    return normalize_bars(raw)


def auto_find_15m_csv(root: str | Path = ".") -> pd.DataFrame | None:
    """
    カレントフォルダ以下から 15m / 15min CSVを探し、
    年別CSVが複数なら結合する。
    """
    root = Path(root)
    patterns = ("*15m*.csv", "*15min*.csv", "*15_min*.csv", "*15M*.csv")
    files = []
    for pattern in patterns:
        files.extend(root.rglob(pattern))
    files = sorted(set(files))

    if not files:
        return None

    print("\n[AUTO DATA] 15分足CSV候補:")
    for p in files[:30]:
        print(" ", p)
    if len(files) > 30:
        print(f"  ... and {len(files)-30} more")

    frames = []
    used = []
    for p in files:
        try:
            f = read_csv_flexible(p)
            # ある程度の長さがあるUSDJPY系のファイルを採用
            if len(f) >= 1000:
                frames.append(f)
                used.append(p)
        except Exception:
            continue

    if not frames:
        return None

    combined = pd.concat(frames).sort_index()

    # 重複timestampが複数年ファイル間にあれば同値だけ許容
    if combined.index.has_duplicates:
        duplicate_groups = combined.loc[
            combined.index.duplicated(keep=False)
        ].groupby(level=0)
        bad = []
        for ts, g in duplicate_groups:
            if len(g.drop_duplicates()) > 1:
                bad.append(ts)
        if bad:
            raise ValueError(f"CSV結合後に異なる重複OHLCがあります: {bad[:3]}")
        combined = combined[~combined.index.duplicated(keep="first")]

    print("\n[AUTO DATA] 使用CSV:")
    for p in used:
        print(" ", p)
    return normalize_bars(combined)


def auto_find_notebook_dataframe(namespace: dict) -> tuple[str, pd.DataFrame] | None:
    """
    Notebookのuser namespaceから、
    15分足OHLCらしいDataFrameを自動検出する。
    """
    candidates = []
    for name, obj in namespace.items():
        if not isinstance(obj, pd.DataFrame) or len(obj) < 1000:
            continue

        cols = {str(c).strip().lower() for c in obj.columns}
        if not {"open", "high", "low", "close"}.issubset(cols):
            continue

        if not isinstance(obj.index, pd.DatetimeIndex):
            continue

        # 最初の数千点で典型間隔を見る
        idx = obj.index[: min(len(obj), 5000)]
        if len(idx) < 20:
            continue
        diffs = pd.Series(idx[1:] - idx[:-1])
        positive = diffs[diffs > pd.Timedelta(0)]
        if positive.empty:
            continue
        median_diff = positive.median()
        if median_diff != pd.Timedelta(minutes=15):
            continue

        candidates.append((len(obj), name, obj))

    if not candidates:
        return None

    candidates.sort(reverse=True, key=lambda x: x[0])
    _, name, obj = candidates[0]
    return name, normalize_bars(obj)


# ============================================================
# 2. 現行30特徴量
# ============================================================

def calculate_rsi(close: pd.Series, period: int = 14) -> pd.Series:
    delta = close.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_gain = gain.rolling(period).mean()
    avg_loss = loss.rolling(period).mean()
    rs = avg_gain / avg_loss.replace(0, np.nan)
    return 100 - 100 / (1 + rs)


def make_features(data: pd.DataFrame) -> pd.DataFrame:
    """
    GitHub現行15分足の30特徴量をそのまま作る。
    未来データを使うcentered rolling等は禁止。
    """
    x = data.copy()

    for n in [1, 2, 4, 8, 16]:
        x[f"return_{n}"] = x["close"].pct_change(n)

    for n in [4, 8, 16, 32]:
        x[f"vol_{n}"] = x["return_1"].rolling(n).std()

    for period in [5, 10, 20, 50, 100]:
        ma = x["close"].rolling(period).mean()
        x[f"ma{period}_distance"] = x["close"] / ma - 1
        x[f"ma{period}_slope"] = ma.pct_change()

    candle_range = (x["high"] - x["low"]).replace(0, np.nan)
    x["body"] = (x["close"] - x["open"]) / candle_range
    x["upper_wick"] = (
        x["high"] - x[["open", "close"]].max(axis=1)
    ) / candle_range
    x["lower_wick"] = (
        x[["open", "close"]].min(axis=1) - x["low"]
    ) / candle_range
    x["range_pct"] = (x["high"] - x["low"]) / x["close"]

    x["rsi14"] = calculate_rsi(x["close"], 14) / 100

    prev_close = x["close"].shift(1)
    true_range = pd.concat(
        [
            x["high"] - x["low"],
            (x["high"] - prev_close).abs(),
            (x["low"] - prev_close).abs(),
        ],
        axis=1,
    ).max(axis=1)
    x["atr14"] = true_range.rolling(14).mean() / x["close"]

    high16 = x["high"].rolling(16).max()
    low16 = x["low"].rolling(16).min()
    x["distance_high_16"] = (high16 - x["close"]) / x["close"]
    x["distance_low_16"] = (x["close"] - low16) / x["close"]

    hour = x.index.hour + x.index.minute / 60
    x["hour_sin"] = np.sin(2 * np.pi * hour / 24)
    x["hour_cos"] = np.cos(2 * np.pi * hour / 24)
    x["weekday"] = x.index.dayofweek / 4

    return x.replace([np.inf, -np.inf], np.nan)


# ============================================================
# 3. 30分先ラベル — signal t -> Open(t+1) -> Close(t+2)
# ============================================================

def prepare_dataset(bars: pd.DataFrame) -> pd.DataFrame:
    x = make_features(bars)

    times = pd.Series(bars.index, index=bars.index)

    x["entry_time"] = times.shift(-1)
    x["label_end"] = times.shift(-2) + pd.Timedelta(minutes=15)

    x["entry_price"] = bars["open"].shift(-1)
    x["exit_price"] = bars["close"].shift(-2)
    x["future_return"] = x["exit_price"] / x["entry_price"] - 1
    x["target"] = (x["future_return"] > 0).astype(int)

    # t,t+1,t+2が本当に連続した15分足か
    continuous = (
        (times.shift(-1) - times).eq(pd.Timedelta(minutes=15))
        & (times.shift(-2) - times).eq(pd.Timedelta(minutes=30))
    )

    required = FEATURES + [
        "entry_time", "label_end", "entry_price", "exit_price", "future_return"
    ]
    out = x.loc[continuous].dropna(subset=required).copy()

    if out.empty:
        raise ValueError("特徴量・連続性チェック後に使用可能データがありません。")

    return out


# ============================================================
# 4. モデル
# ============================================================

def build_model(name: str):
    if name == "RANDOM_FOREST":
        return RandomForestClassifier(**RF_CONFIG)
    if name == "HIST_GRADIENT_BOOSTING":
        return HistGradientBoostingClassifier(**HGB_CONFIG)
    raise ValueError(name)


def fit_model(name: str, train: pd.DataFrame):
    if len(train) < MIN_TRAIN_ROWS:
        raise ValueError("insufficient training rows")
    if train["target"].nunique() < 2:
        raise ValueError("training target has one class")

    model = build_model(name)
    model.fit(train[FEATURES], train["target"])
    return model


def raw_probability(model, frame: pd.DataFrame) -> np.ndarray:
    classes = list(model.classes_)
    if 1 not in classes:
        raise ValueError("Model has no positive class.")
    return model.predict_proba(frame[FEATURES])[:, classes.index(1)]


# ============================================================
# 5. Annual chronological split
# ============================================================

@dataclass(frozen=True)
class AnnualSplit:
    test_year: int
    validation_year: int
    train: pd.DataFrame
    validation: pd.DataFrame
    final_train: pd.DataFrame
    test: pd.DataFrame


def make_annual_split(data: pd.DataFrame, test_year: int) -> AnnualSplit | None:
    val_year = test_year - 1

    years_before_val = sorted(y for y in data.index.year.unique() if y < val_year)
    if len(years_before_val) < MIN_TRAIN_YEARS:
        return None

    val_start = pd.Timestamp(f"{val_year}-01-01", tz="UTC")
    test_start = pd.Timestamp(f"{test_year}-01-01", tz="UTC")
    test_end = pd.Timestamp(f"{test_year + 1}-01-01", tz="UTC")

    train = data.loc[
        (data.index < val_start)
        & (data["label_end"] <= val_start)
    ].copy()

    validation = data.loc[
        (data.index >= val_start)
        & (data.index < test_start)
        & (data["label_end"] <= test_start)
    ].copy()

    final_train = data.loc[
        (data.index < test_start)
        & (data["label_end"] <= test_start)
    ].copy()

    test = data.loc[
        (data.index >= test_start)
        & (data.index < test_end)
        & (data["label_end"] <= test_end)
    ].copy()

    if (
        len(train) < MIN_TRAIN_ROWS
        or len(final_train) < MIN_TRAIN_ROWS
        or len(validation) < MIN_EVAL_ROWS
        or len(test) < MIN_EVAL_ROWS
    ):
        return None

    # 境界リークの明示チェック
    if not (train["label_end"] <= validation.index.min()).all():
        raise AssertionError("Train label crosses Validation boundary.")
    if not (final_train["label_end"] <= test.index.min()).all():
        raise AssertionError("Final-train label crosses Test boundary.")

    return AnnualSplit(
        test_year=test_year,
        validation_year=val_year,
        train=train,
        validation=validation,
        final_train=final_train,
        test=test,
    )


# ============================================================
# 6. OOF Calibration
# ============================================================

def expanding_oof_probabilities(
    model_name: str,
    train_history: pd.DataFrame,
    min_prior_years: int = 2,
) -> pd.DataFrame:
    """
    Calibration学習用の確率は必ずforward/OOFにする。
    例:
      2016-17 -> 2018予測
      2016-18 -> 2019予測
    """
    rows = []
    years = sorted(train_history.index.year.unique())

    for year in years:
        prior_years = [y for y in years if y < year]
        if len(prior_years) < min_prior_years:
            continue

        start = pd.Timestamp(f"{year}-01-01", tz="UTC")
        end = pd.Timestamp(f"{year + 1}-01-01", tz="UTC")

        hist = train_history.loc[
            (train_history.index < start)
            & (train_history["label_end"] <= start)
        ]
        oof = train_history.loc[
            (train_history.index >= start)
            & (train_history.index < end)
            & (train_history["label_end"] <= end)
        ]

        if len(hist) < MIN_TRAIN_ROWS or len(oof) < MIN_EVAL_ROWS:
            continue
        if hist["target"].nunique() < 2:
            continue

        model = fit_model(model_name, hist)
        p = raw_probability(model, oof)

        piece = pd.DataFrame(
            {
                "raw_probability": p,
                "target": oof["target"].to_numpy(dtype=int),
            },
            index=oof.index,
        )
        rows.append(piece)

    if not rows:
        return pd.DataFrame(columns=["raw_probability", "target"])

    return pd.concat(rows).sort_index()


class IdentityCalibrator:
    def fit(self, p, y):
        return self

    def predict(self, p):
        return np.asarray(p, dtype=float)


class PlattCalibrator:
    def __init__(self):
        self.model = LogisticRegression(
            C=1.0,
            solver="lbfgs",
            random_state=RANDOM_SEED,
        )

    def fit(self, p, y):
        self.model.fit(np.asarray(p).reshape(-1, 1), y)
        return self

    def predict(self, p):
        return self.model.predict_proba(
            np.asarray(p).reshape(-1, 1)
        )[:, 1]


class IsotonicCalibrator:
    def __init__(self):
        self.model = IsotonicRegression(
            y_min=0.0,
            y_max=1.0,
            out_of_bounds="clip",
        )

    def fit(self, p, y):
        self.model.fit(np.asarray(p), y)
        return self

    def predict(self, p):
        return np.asarray(self.model.predict(np.asarray(p)), dtype=float)


def fit_calibrator(method: str, oof: pd.DataFrame):
    if method == "RAW":
        return IdentityCalibrator()

    if len(oof) < 500 or oof["target"].nunique() < 2:
        # Calibrationに十分なforward sampleが無ければRAWへ
        return IdentityCalibrator()

    if method == "PLATT":
        return PlattCalibrator().fit(
            oof["raw_probability"].to_numpy(),
            oof["target"].to_numpy(),
        )

    if method == "ISOTONIC":
        # Isotonicは極端な小標本に弱いので最低1000
        if len(oof) < 1000:
            return IdentityCalibrator()
        return IsotonicCalibrator().fit(
            oof["raw_probability"].to_numpy(),
            oof["target"].to_numpy(),
        )

    raise ValueError(method)


def expected_calibration_error(
    y_true: np.ndarray,
    p: np.ndarray,
    bins: int = 10,
) -> float:
    y = np.asarray(y_true, dtype=float)
    p = np.asarray(p, dtype=float)
    edges = np.linspace(0, 1, bins + 1)
    ids = np.clip(np.digitize(p, edges[1:-1], right=False), 0, bins - 1)

    total = len(y)
    if total == 0:
        return np.nan

    ece = 0.0
    for b in range(bins):
        mask = ids == b
        if not mask.any():
            continue
        ece += mask.mean() * abs(y[mask].mean() - p[mask].mean())
    return float(ece)


def calibration_metrics(y_true, p) -> dict:
    y = np.asarray(y_true, dtype=int)
    p = np.clip(np.asarray(p, dtype=float), 1e-8, 1 - 1e-8)

    auc = roc_auc_score(y, p) if len(np.unique(y)) == 2 else np.nan
    return {
        "brier": float(brier_score_loss(y, p)),
        "ece": expected_calibration_error(y, p, bins=10),
        "auc": float(auc),
    }


def choose_calibration(
    model_name: str,
    train: pd.DataFrame,
    validation: pd.DataFrame,
):
    """
    Calibration methodはValidationのBrier最小で決める。
    経済損益を見てCalibration法を選ばないことで自由度を抑える。
    """
    oof = expanding_oof_probabilities(model_name, train)
    model = fit_model(model_name, train)
    raw_val = raw_probability(model, validation)

    rows = []
    fitted = {}

    for method in CALIBRATION_METHODS:
        cal = fit_calibrator(method, oof)
        p = np.clip(cal.predict(raw_val), 0, 1)
        metrics = calibration_metrics(validation["target"].to_numpy(), p)
        rows.append({"method": method, **metrics})
        fitted[method] = cal

    table = pd.DataFrame(rows).sort_values(["brier", "ece", "method"])
    chosen = str(table.iloc[0]["method"])

    return chosen, table, raw_val


# ============================================================
# 7. Threshold / Session
# ============================================================

def session_mask(index: pd.DatetimeIndex, policy: str) -> np.ndarray:
    hour = index.hour

    if policy == "ALL":
        return np.ones(len(index), dtype=bool)
    if policy == "UTC_13_24":
        return (hour >= 13) & (hour < 24)
    if policy == "UTC_21_24":
        return (hour >= 21) & (hour < 24)
    if policy == "EXCLUDE_08_13":
        return ~((hour >= 8) & (hour < 13))

    raise ValueError(policy)


def predictions_frame(
    frame: pd.DataFrame,
    calibrated_p_up: np.ndarray,
    raw_p_up: np.ndarray | None = None,
) -> pd.DataFrame:
    p = np.asarray(calibrated_p_up, dtype=float)
    direction_sign = np.where(p >= 0.5, 1.0, -1.0)

    out = frame[
        ["entry_time", "label_end", "entry_price", "exit_price", "future_return", "target"]
    ].copy()

    out["p_up"] = p
    out["raw_p_up"] = p if raw_p_up is None else np.asarray(raw_p_up, dtype=float)
    out["confidence"] = np.maximum(p, 1 - p)
    out["direction"] = np.where(p >= 0.5, "BUY", "SELL")
    out["direction_correct"] = (
        (p >= 0.5) == (frame["future_return"].to_numpy() > 0)
    )
    out["gross_return"] = (
        frame["future_return"].to_numpy(dtype=float) * direction_sign
    )
    return out


def select_trades(
    predictions: pd.DataFrame,
    threshold: float,
    session_policy: str,
) -> pd.DataFrame:
    """
    Threshold + Session適用後に、1ポジションずつ非重複で選ぶ。
    """
    mask = (
        (predictions["confidence"] >= threshold)
        & session_mask(predictions.index, session_policy)
    )
    candidates = predictions.loc[mask].sort_index()

    selected = []
    next_free_time = None

    for row in candidates.itertuples():
        # entry_time が既存ポジションの決済時刻より前なら見送る
        if next_free_time is not None and row.entry_time < next_free_time:
            continue
        selected.append(row.Index)
        next_free_time = row.label_end

    trades = candidates.loc[selected].copy()
    trades.index.name = "signal_time"
    trades["base_net_return"] = trades["gross_return"] - COST
    return trades


def basic_stats(returns) -> dict:
    r = np.asarray(returns, dtype=float)
    r = r[np.isfinite(r)]

    if len(r) == 0:
        return {
            "trades": 0,
            "win_rate": np.nan,
            "avg_return": np.nan,
            "median_return": np.nan,
            "profit_factor": np.nan,
            "growth": 0.0,
            "max_dd": np.nan,
            "return_to_dd": np.nan,
        }

    gains = r[r > 0].sum()
    losses = -r[r < 0].sum()
    if losses > 0:
        pf = gains / losses
    elif gains > 0:
        pf = np.inf
    else:
        pf = np.nan

    equity = np.r_[1.0, np.cumprod(1 + r)]
    peak = np.maximum.accumulate(equity)
    dd = equity / peak - 1
    max_dd = float(dd.min())
    growth = float(equity[-1] - 1)

    return_to_dd = (
        growth / abs(max_dd)
        if np.isfinite(max_dd) and max_dd < 0
        else np.nan
    )

    return {
        "trades": int(len(r)),
        "win_rate": float((r > 0).mean()),
        "avg_return": float(r.mean()),
        "median_return": float(np.median(r)),
        "profit_factor": float(pf),
        "growth": growth,
        "max_dd": max_dd,
        "return_to_dd": float(return_to_dd),
    }


def choose_threshold_and_session(
    validation_predictions: pd.DataFrame,
):
    """
    FIXED 1xの状態で Threshold / Session をValidationだけから選ぶ。

    score = avg_return * sqrt(trades)
    これは現在のGitHub baselineの考え方を維持。
    """
    rows = []
    best = None
    best_key = None

    for threshold in THRESHOLDS:
        for session in SESSION_POLICIES:
            trades = select_trades(validation_predictions, threshold, session)
            stats = basic_stats(trades["base_net_return"])
            eligible = stats["trades"] >= MIN_VALIDATION_TRADES

            score = (
                stats["avg_return"] * math.sqrt(stats["trades"])
                if eligible and np.isfinite(stats["avg_return"])
                else np.nan
            )

            row = {
                "threshold": threshold,
                "session": session,
                "eligible": eligible,
                "score": score,
                **stats,
            }
            rows.append(row)

            if not eligible:
                continue

            # tie-break:
            # 1) score
            # 2) PF
            # 3) 多い取引数
            # 4) 低いthreshold（過度な選別を避ける）
            key = (
                float(score),
                float(stats["profit_factor"]) if np.isfinite(stats["profit_factor"]) else -np.inf,
                int(stats["trades"]),
                -float(threshold),
            )
            if best_key is None or key > best_key:
                best_key = key
                best = (float(threshold), str(session))

    table = pd.DataFrame(rows)

    if best is None:
        raise RuntimeError(
            "Validationで最低取引数を満たすThreshold/Sessionがありません。"
            "Testを見てMIN_VALIDATION_TRADESを下げないでください。"
        )

    return best[0], best[1], table


# ============================================================
# 8. Confidence-based Position Sizing
# ============================================================

def raw_position_size(
    confidence: np.ndarray,
    threshold: float,
    policy: str,
) -> np.ndarray:
    """
    Confidenceがthresholdをどれだけ超えたかを0..1へ正規化。
    ここでのSizingはレバレッジ最適化ではなく「配分」の検証。
    """
    c = np.asarray(confidence, dtype=float)
    denom = max(1.0 - threshold, 1e-8)
    edge = np.clip((c - threshold) / denom, 0, 1)

    if policy == "FIXED":
        raw = np.ones_like(edge)
    elif policy == "GENTLE":
        raw = 0.85 + 0.30 * edge       # 0.85 .. 1.15
    elif policy == "MODERATE":
        raw = 0.70 + 0.60 * edge       # 0.70 .. 1.30
    elif policy == "STRONG":
        raw = 0.50 + 1.00 * edge       # 0.50 .. 1.50
    else:
        raise ValueError(policy)

    return raw


def sizing_candidate_stats(
    trades: pd.DataFrame,
    threshold: float,
    policy: str,
):
    if trades.empty:
        return None

    raw = raw_position_size(
        trades["confidence"].to_numpy(),
        threshold,
        policy,
    )

    # Validation内の平均exposureを1.0にして、
    # 単なる「平均レバレッジ増加」と配分alphaを分離。
    scale = 1.0 / raw.mean()
    size = raw * scale

    sized_return = size * (
        trades["gross_return"].to_numpy(dtype=float) - COST
    )
    stats = basic_stats(sized_return)

    return {
        "policy": policy,
        "validation_scale": float(scale),
        "mean_size": float(size.mean()),
        **stats,
    }


def choose_sizing(
    validation_trades: pd.DataFrame,
    threshold: float,
):
    """
    SizingはThreshold/Sessionが決まった後に別段階で選ぶ。
    Adaptiveを採用するにはFIXEDに対して
       Avg Return, PF, Return/DD
    の3つ全てで悪化しないことを要求する。
    """
    rows = []
    for policy in SIZING_POLICIES:
        row = sizing_candidate_stats(validation_trades, threshold, policy)
        if row is not None:
            rows.append(row)

    table = pd.DataFrame(rows)
    if table.empty:
        raise RuntimeError("Sizing validation trades are empty.")

    fixed = table.loc[table["policy"] == "FIXED"].iloc[0]
    candidates = []

    for _, row in table.iterrows():
        if row["policy"] == "FIXED":
            continue

        avg_ok = row["avg_return"] >= fixed["avg_return"]
        pf_ok = row["profit_factor"] >= fixed["profit_factor"]
        rdd_ok = (
            np.isfinite(row["return_to_dd"])
            and np.isfinite(fixed["return_to_dd"])
            and row["return_to_dd"] >= fixed["return_to_dd"]
        )

        if avg_ok and pf_ok and rdd_ok:
            candidates.append(row)

    if not candidates:
        chosen = fixed
    else:
        chosen = max(
            candidates,
            key=lambda r: (
                r["return_to_dd"],
                r["profit_factor"],
                r["avg_return"],
            ),
        )

    return (
        str(chosen["policy"]),
        float(chosen["validation_scale"]),
        table,
    )


def apply_sizing(
    trades: pd.DataFrame,
    threshold: float,
    policy: str,
    validation_scale: float,
    cost_multiplier: float = 1.0,
) -> pd.DataFrame:
    out = trades.copy()

    raw = raw_position_size(
        out["confidence"].to_numpy(),
        threshold,
        policy,
    )
    size = raw * validation_scale

    # 異常なexposureを防ぐ最終安全cap。
    # Validation scaleをTest平均に合わせ直すことはしない。
    size = np.clip(size, 0.25, 2.0)

    out["position_size"] = size
    out["cost_multiplier"] = cost_multiplier
    out["net_return_1x"] = out["gross_return"] - COST * cost_multiplier
    out["sized_return"] = size * out["net_return_1x"]

    return out


# ============================================================
# 9. 1年・1モデルを完全Nestedで評価
# ============================================================

def evaluate_one_year_model(
    model_name: str,
    split: AnnualSplit,
):
    # ---------- A. Validation用モデル ----------
    chosen_calibration, cal_table, raw_val = choose_calibration(
        model_name,
        split.train,
        split.validation,
    )

    # Calibration methodを固定
    oof_train = expanding_oof_probabilities(model_name, split.train)
    val_calibrator = fit_calibrator(chosen_calibration, oof_train)
    p_val = np.clip(val_calibrator.predict(raw_val), 0, 1)

    val_predictions = predictions_frame(
        split.validation,
        p_val,
        raw_val,
    )

    # ---------- B. Threshold + Session ----------
    threshold, session, threshold_table = choose_threshold_and_session(
        val_predictions
    )

    validation_selected = select_trades(
        val_predictions,
        threshold,
        session,
    )

    # ---------- C. Sizing ----------
    sizing_policy, validation_scale, sizing_table = choose_sizing(
        validation_selected,
        threshold,
    )

    # ---------- D. Test前に全historyへ再学習 ----------
    # Calibrationは選択済みmethodを維持し、
    # final_train上のforward/OOF確率から再fitする。
    final_oof = expanding_oof_probabilities(
        model_name,
        split.final_train,
    )
    final_calibrator = fit_calibrator(
        chosen_calibration,
        final_oof,
    )

    final_model = fit_model(
        model_name,
        split.final_train,
    )
    raw_test = raw_probability(
        final_model,
        split.test,
    )
    p_test = np.clip(
        final_calibrator.predict(raw_test),
        0,
        1,
    )

    test_predictions = predictions_frame(
        split.test,
        p_test,
        raw_test,
    )

    selected = select_trades(
        test_predictions,
        threshold,
        session,
    )

    final_trades = apply_sizing(
        selected,
        threshold,
        sizing_policy,
        validation_scale,
        cost_multiplier=1.0,
    )

    stats = basic_stats(final_trades["sized_return"])

    test_cal_metrics = calibration_metrics(
        split.test["target"].to_numpy(),
        p_test,
    )
    raw_auc = (
        roc_auc_score(split.test["target"], raw_test)
        if split.test["target"].nunique() == 2
        else np.nan
    )

    result = {
        "test_year": split.test_year,
        "validation_year": split.validation_year,
        "model": model_name,
        "calibration": chosen_calibration,
        "threshold": threshold,
        "session": session,
        "sizing_policy": sizing_policy,
        "validation_scale": validation_scale,
        "raw_auc": float(raw_auc),
        "calibrated_auc": test_cal_metrics["auc"],
        "test_brier": test_cal_metrics["brier"],
        "test_ece": test_cal_metrics["ece"],
        "mean_position_size": (
            float(final_trades["position_size"].mean())
            if len(final_trades)
            else np.nan
        ),
        **stats,
    }

    # 出力用tableへyear/modelを付与
    cal_table = cal_table.copy()
    cal_table["test_year"] = split.test_year
    cal_table["model"] = model_name
    cal_table["selected"] = cal_table["method"].eq(chosen_calibration)

    threshold_table = threshold_table.copy()
    threshold_table["test_year"] = split.test_year
    threshold_table["model"] = model_name
    threshold_table["selected"] = (
        threshold_table["threshold"].eq(threshold)
        & threshold_table["session"].eq(session)
    )

    sizing_table = sizing_table.copy()
    sizing_table["test_year"] = split.test_year
    sizing_table["model"] = model_name
    sizing_table["selected"] = sizing_table["policy"].eq(sizing_policy)

    final_trades = final_trades.copy()
    final_trades["test_year"] = split.test_year
    final_trades["model"] = model_name
    final_trades["calibration"] = chosen_calibration
    final_trades["threshold"] = threshold
    final_trades["session"] = session
    final_trades["sizing_policy"] = sizing_policy

    return (
        result,
        final_trades,
        cal_table,
        threshold_table,
        sizing_table,
    )


# ============================================================
# 10. Cost stress
# ============================================================

def cost_stress_table(trades: pd.DataFrame) -> pd.DataFrame:
    rows = []

    if trades.empty:
        return pd.DataFrame()

    for model in MODEL_NAMES:
        m = trades.loc[trades["model"] == model].copy()
        if m.empty:
            continue

        for mult in COST_STRESS_MULTIPLIERS:
            r = m["position_size"].to_numpy() * (
                m["gross_return"].to_numpy() - COST * mult
            )
            stats = basic_stats(r)
            rows.append(
                {
                    "model": model,
                    "cost_multiplier": mult,
                    "round_trip_cost": COST * mult,
                    **stats,
                }
            )

    return pd.DataFrame(rows)


# ============================================================
# 11. Confidence bands
# ============================================================

def confidence_band_table(trades: pd.DataFrame) -> pd.DataFrame:
    if trades.empty:
        return pd.DataFrame()

    edges = [0.50, 0.52, 0.54, 0.56, 0.58, 0.60, 0.62, 0.65, 0.70, 1.000001]
    labels = [
        "50-52", "52-54", "54-56", "56-58", "58-60",
        "60-62", "62-65", "65-70", "70+",
    ]

    x = trades.copy()
    x["confidence_band"] = pd.cut(
        x["confidence"],
        bins=edges,
        labels=labels,
        right=False,
    )

    rows = []
    for model in MODEL_NAMES:
        m = x.loc[x["model"] == model]
        for band in labels:
            b = m.loc[m["confidence_band"] == band]
            if b.empty:
                continue

            stats = basic_stats(b["sized_return"])
            rows.append(
                {
                    "model": model,
                    "confidence_band": band,
                    "mean_confidence": float(b["confidence"].mean()),
                    **stats,
                }
            )

    return pd.DataFrame(rows)


# ============================================================
# 12. BUY / SELL
# ============================================================

def side_table(trades: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for model in MODEL_NAMES:
        m = trades.loc[trades["model"] == model]
        for side in ("ALL", "BUY", "SELL"):
            s = m if side == "ALL" else m.loc[m["direction"] == side]
            stats = basic_stats(s["sized_return"])
            rows.append({"model": model, "side": side, **stats})
    return pd.DataFrame(rows)


# ============================================================
# 13. Daily paired moving-block bootstrap HGB vs RF
# ============================================================

def daily_returns(trades: pd.DataFrame) -> pd.DataFrame:
    if trades.empty:
        return pd.DataFrame()

    x = trades.copy()
    x["date"] = pd.DatetimeIndex(x.index).normalize()

    rows = []
    for (model, date), g in x.groupby(["model", "date"]):
        daily = float(np.prod(1 + g["sized_return"].to_numpy()) - 1)
        rows.append({"model": model, "date": date, "daily_return": daily})

    return pd.DataFrame(rows)


def moving_block_bootstrap_alpha(
    development_trades: pd.DataFrame,
    iterations: int = BOOTSTRAP_ITERATIONS,
    block_days: int = BOOTSTRAP_BLOCK_DAYS,
) -> dict:
    d = daily_returns(development_trades)
    if d.empty:
        return {}

    pivot = d.pivot(index="date", columns="model", values="daily_return").fillna(0.0)

    if not set(MODEL_NAMES).issubset(pivot.columns):
        return {}

    delta = (
        pivot["HIST_GRADIENT_BOOSTING"]
        - pivot["RANDOM_FOREST"]
    ).to_numpy(dtype=float)

    n = len(delta)
    if n < max(30, block_days * 3):
        return {}

    rng = np.random.default_rng(RANDOM_SEED)
    starts = np.arange(max(1, n - block_days + 1))
    means = np.empty(iterations, dtype=float)

    for i in range(iterations):
        sample = []
        while len(sample) < n:
            s = int(rng.choice(starts))
            sample.extend(delta[s : s + block_days])
        sample = np.asarray(sample[:n], dtype=float)
        means[i] = sample.mean()

    observed = float(delta.mean())
    low, high = np.percentile(means, [2.5, 97.5])
    prob = float((means > 0).mean())

    return {
        "observed_daily_alpha": observed,
        "ci_2_5": float(low),
        "ci_97_5": float(high),
        "prob_alpha_positive": prob,
        "days": int(n),
        "iterations": int(iterations),
        "block_days": int(block_days),
    }


# ============================================================
# 14. Aggregate summary / decision
# ============================================================

def aggregate_model_stats(trades: pd.DataFrame, years: tuple[int, ...]) -> pd.DataFrame:
    rows = []
    selected = trades.loc[trades["test_year"].isin(years)]

    for model in MODEL_NAMES:
        m = selected.loc[selected["model"] == model].sort_index()
        stats = basic_stats(m["sized_return"])
        rows.append({"model": model, **stats})
    return pd.DataFrame(rows)


def selection_frequency(annual: pd.DataFrame, column: str) -> pd.DataFrame:
    if annual.empty:
        return pd.DataFrame()
    return (
        annual.groupby(["model", column])
        .size()
        .rename("count")
        .reset_index()
        .sort_values(["model", "count"], ascending=[True, False])
    )


def make_final_decision(
    annual: pd.DataFrame,
    dev_summary: pd.DataFrame,
    bootstrap: dict,
) -> str:
    if dev_summary.empty:
        return "INSUFFICIENT_RESULTS"

    by_model = dev_summary.set_index("model")
    if not set(MODEL_NAMES).issubset(by_model.index):
        return "INSUFFICIENT_RESULTS"

    hgb = by_model.loc["HIST_GRADIENT_BOOSTING"]
    rf = by_model.loc["RANDOM_FOREST"]

    dev = annual.loc[
        annual["test_year"].isin(DEVELOPMENT_TEST_YEARS)
    ]
    hgb_year = dev.loc[dev["model"] == "HIST_GRADIENT_BOOSTING"]

    positive_years = int((hgb_year["avg_return"] > 0).sum())
    pf_years = int((hgb_year["profit_factor"] > 1).sum())
    evaluated_years = len(hgb_year)

    alpha_prob = bootstrap.get("prob_alpha_positive", np.nan)

    confirmation = annual.loc[
        (annual["test_year"] == CONFIRMATION_YEAR)
        & (annual["model"] == "HIST_GRADIENT_BOOSTING")
    ]

    confirmation_ok = False
    if len(confirmation):
        c = confirmation.iloc[0]
        confirmation_ok = (
            c["avg_return"] > 0
            and c["profit_factor"] > 1
            and c["max_dd"] < 0
        )

    dev_robust = (
        evaluated_years >= 5
        and positive_years >= max(4, evaluated_years - 1)
        and pf_years >= max(4, evaluated_years - 1)
        and hgb["avg_return"] > 0
        and hgb["profit_factor"] > 1
    )

    beats_rf = (
        hgb["avg_return"] > rf["avg_return"]
        and hgb["profit_factor"] > rf["profit_factor"]
        and (
            not np.isfinite(hgb["return_to_dd"])
            or not np.isfinite(rf["return_to_dd"])
            or hgb["return_to_dd"] > rf["return_to_dd"]
        )
    )

    bootstrap_ok = (
        np.isfinite(alpha_prob)
        and alpha_prob >= 0.80
    )

    if dev_robust and beats_rf and bootstrap_ok and confirmation_ok:
        return "LOCK_HGB_AS_PROVISIONAL_CHAMPION_AND_MOVE_TO_FEATURE_TOURNAMENT"

    if dev_robust and beats_rf and confirmation_ok:
        return "KEEP_HGB_AS_PROVISIONAL_CHAMPION_BUT_RF_REMAINS_STRONG_BENCHMARK"

    if dev_robust:
        return "HGB_IS_ROBUST_BUT_NOT_CLEARLY_BETTER_THAN_RF"

    return "DO_NOT_LOCK_HGB_YET"


# ============================================================
# 15. Main experiment
# ============================================================

@dataclass
class RunResult:
    output_dir: Path
    annual: pd.DataFrame
    development_summary: pd.DataFrame
    confirmation_summary: pd.DataFrame
    bootstrap: dict
    decision: str


def run_experiment(
    bars: pd.DataFrame,
    output_root: str | Path = "hgb_champion_runs",
) -> RunResult:
    bars = normalize_bars(bars)
    data = prepare_dataset(bars)

    print("\n" + "=" * 90)
    print("HGB CHAMPION LOCK-IN")
    print("=" * 90)
    print(f"Bars: {len(bars):,}")
    print(f"Usable rows: {len(data):,}")
    print(f"Period: {bars.index.min()} -> {bars.index.max()}")
    print(f"Features: {len(FEATURES)}")
    print(f"Target UP rate: {data['target'].mean():.4f}")
    print(f"Cost: {COST:.6f} ({COST*100:.4f}%)")
    print("Exit: fixed 30 minutes")
    print("Risk Engine: NO_RISK")
    print("Development OOS:", DEVELOPMENT_TEST_YEARS)
    print("Confirmation year:", CONFIRMATION_YEAR)

    years_available = set(data.index.year.unique())
    test_years = [
        y for y in list(DEVELOPMENT_TEST_YEARS) + [CONFIRMATION_YEAR]
        if y in years_available
    ]

    if not test_years:
        raise ValueError("2020-2026の評価年がデータにありません。")

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    out = Path(output_root) / f"hgb_lockin_{timestamp}"
    out.mkdir(parents=True, exist_ok=False)

    annual_rows = []
    trade_frames = []
    cal_frames = []
    threshold_frames = []
    sizing_frames = []

    for year in test_years:
        split = make_annual_split(data, year)
        if split is None:
            print(f"\nTEST YEAR {year}: SKIPPED (insufficient chronological data)")
            continue

        print("\n" + "=" * 90)
        print(f"TEST YEAR {year}")
        print(
            f"Train<{split.validation_year}: {len(split.train):,} | "
            f"Validation={split.validation_year}: {len(split.validation):,} | "
            f"Final train<{year}: {len(split.final_train):,} | "
            f"Test={year}: {len(split.test):,}"
        )

        for model_name in MODEL_NAMES:
            print("\n" + "-" * 80)
            print("Running:", model_name)

            try:
                (
                    result,
                    trades,
                    cal_table,
                    threshold_table,
                    sizing_table,
                ) = evaluate_one_year_model(model_name, split)

                annual_rows.append(result)
                trade_frames.append(trades)
                cal_frames.append(cal_table)
                threshold_frames.append(threshold_table)
                sizing_frames.append(sizing_table)

                print(
                    f"Calibration={result['calibration']} | "
                    f"Threshold={result['threshold']:.2f} | "
                    f"Session={result['session']} | "
                    f"Sizing={result['sizing_policy']}"
                )
                print(
                    f"Trades={result['trades']} | "
                    f"PF={result['profit_factor']:.3f} | "
                    f"Avg={result['avg_return']*100:.5f}% | "
                    f"Growth={result['growth']*100:.3f}% | "
                    f"MaxDD={result['max_dd']*100:.3f}% | "
                    f"Return/DD={result['return_to_dd']:.3f}"
                )
                print(
                    f"Raw AUC={result['raw_auc']:.4f} | "
                    f"Cal AUC={result['calibrated_auc']:.4f} | "
                    f"Brier={result['test_brier']:.6f} | "
                    f"ECE={result['test_ece']:.6f}"
                )

            except Exception as exc:
                print(f"FAILED: {type(exc).__name__}: {exc}")
                annual_rows.append(
                    {
                        "test_year": year,
                        "validation_year": year - 1,
                        "model": model_name,
                        "status": "FAILED",
                        "error": f"{type(exc).__name__}: {exc}",
                    }
                )

    annual = pd.DataFrame(annual_rows)
    trades = (
        pd.concat(trade_frames).sort_index()
        if trade_frames
        else pd.DataFrame()
    )
    cal = pd.concat(cal_frames, ignore_index=True) if cal_frames else pd.DataFrame()
    threshold_search = (
        pd.concat(threshold_frames, ignore_index=True)
        if threshold_frames
        else pd.DataFrame()
    )
    sizing_search = (
        pd.concat(sizing_frames, ignore_index=True)
        if sizing_frames
        else pd.DataFrame()
    )

    # FAILED行を除いた安全な集計
    evaluated = annual.loc[
        annual.get("trades", pd.Series(index=annual.index, dtype=float)).notna()
    ].copy()

    dev_summary = aggregate_model_stats(
        trades,
        DEVELOPMENT_TEST_YEARS,
    )
    confirmation_summary = aggregate_model_stats(
        trades,
        (CONFIRMATION_YEAR,),
    )

    dev_trades = trades.loc[
        trades["test_year"].isin(DEVELOPMENT_TEST_YEARS)
    ] if not trades.empty else pd.DataFrame()

    bootstrap = moving_block_bootstrap_alpha(dev_trades)

    cost_stress = cost_stress_table(dev_trades)
    confidence_bands = confidence_band_table(dev_trades)
    sides = side_table(dev_trades)

    calibration_freq = selection_frequency(evaluated, "calibration")
    session_freq = selection_frequency(evaluated, "session")
    sizing_freq = selection_frequency(evaluated, "sizing_policy")

    decision = make_final_decision(
        evaluated,
        dev_summary,
        bootstrap,
    )

    # ---------- 保存 ----------
    annual.to_csv(out / "annual_results.csv", index=False)
    dev_summary.to_csv(out / "development_summary_2020_2025.csv", index=False)
    confirmation_summary.to_csv(out / "confirmation_2026.csv", index=False)
    cost_stress.to_csv(out / "cost_stress.csv", index=False)
    confidence_bands.to_csv(out / "confidence_bands.csv", index=False)
    sides.to_csv(out / "buy_sell.csv", index=False)
    cal.to_csv(out / "calibration_search.csv", index=False)
    threshold_search.to_csv(out / "threshold_session_search.csv", index=False)
    sizing_search.to_csv(out / "sizing_search.csv", index=False)
    calibration_freq.to_csv(out / "calibration_frequency.csv", index=False)
    session_freq.to_csv(out / "session_frequency.csv", index=False)
    sizing_freq.to_csv(out / "sizing_frequency.csv", index=False)

    if not trades.empty:
        trades.to_csv(out / "all_oos_trades.csv", index_label="signal_time")

    pd.DataFrame([bootstrap]).to_csv(
        out / "paired_block_bootstrap_hgb_vs_rf.csv",
        index=False,
    )

    metadata = {
        "experiment": "hgb-champion-lockin-v1",
        "features": FEATURES,
        "cost": COST,
        "development_test_years": DEVELOPMENT_TEST_YEARS,
        "confirmation_year": CONFIRMATION_YEAR,
        "rf_config": RF_CONFIG,
        "hgb_config": HGB_CONFIG,
        "thresholds": THRESHOLDS,
        "sessions": SESSION_POLICIES,
        "sizing_policies": SIZING_POLICIES,
        "calibration_methods": CALIBRATION_METHODS,
        "decision": decision,
        "notes": [
            "2026 is confirmation, not a pristine holdout, because it has already been inspected.",
            "No TP/SL optimization in this experiment.",
            "Risk policy fixed to NO_RISK.",
            "Feature set frozen at 30 features. Feature engineering starts only after this lock-in.",
            "Position sizing uses validation-normalized exposure; test mean exposure is not renormalized.",
        ],
    }
    (out / "run_config.json").write_text(
        json.dumps(metadata, ensure_ascii=False, indent=2, default=str),
        encoding="utf-8",
    )
    (out / "FINAL_DECISION.txt").write_text(decision + "\n", encoding="utf-8")

    # ---------- 画面表示 ----------
    print("\n" + "=" * 90)
    print("DEVELOPMENT OOS SUMMARY: 2020-2025")
    print("=" * 90)
    if not dev_summary.empty:
        show = dev_summary.copy()
        for c in ["win_rate", "avg_return", "median_return", "growth", "max_dd"]:
            if c in show:
                show[c] = show[c] * 100
        print(show.to_string(index=False))

    print("\n" + "=" * 90)
    print("CONFIRMATION: 2026")
    print("=" * 90)
    if not confirmation_summary.empty:
        show = confirmation_summary.copy()
        for c in ["win_rate", "avg_return", "median_return", "growth", "max_dd"]:
            if c in show:
                show[c] = show[c] * 100
        print(show.to_string(index=False))

    print("\n" + "=" * 90)
    print("SELECTION STABILITY")
    print("=" * 90)
    print("\nCalibration frequency")
    print(calibration_freq.to_string(index=False) if not calibration_freq.empty else "N/A")
    print("\nSession frequency")
    print(session_freq.to_string(index=False) if not session_freq.empty else "N/A")
    print("\nSizing frequency")
    print(sizing_freq.to_string(index=False) if not sizing_freq.empty else "N/A")

    print("\n" + "=" * 90)
    print("COST STRESS: DEVELOPMENT OOS")
    print("=" * 90)
    if not cost_stress.empty:
        show = cost_stress.copy()
        for c in ["win_rate", "avg_return", "median_return", "growth", "max_dd"]:
            if c in show:
                show[c] = show[c] * 100
        print(show.to_string(index=False))

    print("\n" + "=" * 90)
    print("PAIRED MOVING-BLOCK BOOTSTRAP: HGB - RF")
    print("=" * 90)
    if bootstrap:
        print(
            f"Observed daily alpha: {bootstrap['observed_daily_alpha']*100:.6f}%\n"
            f"95% CI: {bootstrap['ci_2_5']*100:.6f}% "
            f"~ {bootstrap['ci_97_5']*100:.6f}%\n"
            f"P(alpha > 0): {bootstrap['prob_alpha_positive']*100:.2f}%\n"
            f"Days: {bootstrap['days']}"
        )
    else:
        print("Bootstrap unavailable.")

    print("\n" + "=" * 90)
    print("FINAL DECISION")
    print("=" * 90)
    print(decision)

    if decision.startswith("LOCK_HGB"):
        print(
            "\n次: HGBモデル・30分Exit・NO_RISKを固定して、"
            "Feature Tournamentへ進んでよい候補です。"
        )
    elif "PROVISIONAL_CHAMPION" in decision:
        print(
            "\nHGBは暫定Championのまま維持。ただしRFをBenchmarkとして残し、"
            "Feature Tournamentでは両方を監視してください。"
        )
    else:
        print(
            "\nFeature追加前にannual_results / selection stability / cost stressを確認し、"
            "HGB固定の根拠が弱い原因を診断してください。"
        )

    print("\nSaved to:", out.resolve())

    return RunResult(
        output_dir=out,
        annual=annual,
        development_summary=dev_summary,
        confirmation_summary=confirmation_summary,
        bootstrap=bootstrap,
        decision=decision,
    )


# ============================================================
# 16. Jupyter便利関数
# ============================================================

def run_from_dataframe(
    frame: pd.DataFrame,
    output_root: str | Path = "hgb_champion_runs",
):
    return run_experiment(frame, output_root=output_root)


def run_from_notebook(
    namespace: dict | None = None,
    output_root: str | Path = "hgb_champion_runs",
):
    if namespace is None:
        try:
            ip = get_ipython()  # noqa: F821
            namespace = ip.user_ns
        except Exception:
            namespace = globals()

    found = auto_find_notebook_dataframe(namespace)
    if found is None:
        print("Notebook内に15分足OHLC DataFrameを自動検出できませんでした。")
        bars = auto_find_15m_csv(".")
        if bars is None:
            raise FileNotFoundError(
                "15分足DataFrameも *15m*.csv も見つかりません。"
                "run_from_dataframe(your_dataframe) を使うか --csv を指定してください。"
            )
        return run_experiment(bars, output_root=output_root)

    name, bars = found
    print(f"[AUTO DATA] Notebook DataFrame '{name}' を使用します: {len(bars):,} rows")
    return run_experiment(bars, output_root=output_root)


# ============================================================
# 17. CLI / %run
# ============================================================

def main():
    parser = argparse.ArgumentParser(add_help=True)
    parser.add_argument(
        "--csv",
        type=str,
        default=None,
        help="15分足CSV。未指定ならNotebook DataFrame/15m CSVを自動探索。",
    )
    parser.add_argument(
        "--out",
        type=str,
        default="hgb_champion_runs",
        help="結果保存先の親フォルダ",
    )

    # Jupyter %run ではIPython由来引数を無視できるようparse_known_args
    args, _unknown = parser.parse_known_args()

    if args.csv:
        bars = read_csv_flexible(args.csv)
        run_experiment(bars, output_root=args.out)
        return

    # Notebookならuser_nsを探す
    try:
        ip = get_ipython()  # noqa: F821
        ns = ip.user_ns
    except Exception:
        ns = None

    if ns is not None:
        found = auto_find_notebook_dataframe(ns)
        if found is not None:
            name, bars = found
            print(f"[AUTO DATA] Notebook DataFrame '{name}' を使用: {len(bars):,} rows")
            run_experiment(bars, output_root=args.out)
            return

    # 最後にカレントディレクトリの15m CSVを探す
    bars = auto_find_15m_csv(".")
    if bars is None:
        raise FileNotFoundError(
            "15分足データを自動検出できませんでした。\n"
            "Notebookなら:\n"
            "  from hgb_champion_lockin import run_from_dataframe\n"
            "  result = run_from_dataframe(あなたの15分足DataFrame)\n"
            "CSVなら:\n"
            '  python hgb_champion_lockin.py --csv "path/to/usdjpy_15m.csv"'
        )

    run_experiment(bars, output_root=args.out)


if __name__ == "__main__":
    main()


## 元セルindex 52


In [ ]:
import numpy as np
import pandas as pd


# ============================================================
# 1. OHLC DataFrameを自動で探す
# ============================================================

def find_ohlc_dataframe():
    """
    Jupyter Notebook内から
    Open / High / Low / Close を持つDataFrameを探す。

    15分足があれば15分足を優先。
    なければ5分足を使う。
    """

    candidates = []

    for name, obj in globals().items():

        if not isinstance(obj, pd.DataFrame):
            continue

        if len(obj) < 100:
            continue

        # 列名を小文字にして判定
        cols = {str(c).lower() for c in obj.columns}

        if not {"open", "high", "low", "close"}.issubset(cols):
            continue

        if not isinstance(obj.index, pd.DatetimeIndex):
            continue

        # 時系列順に並んでいるものだけ
        x = obj.sort_index()

        # 典型的な足間隔を調べる
        diffs = x.index.to_series().diff().dropna()

        if len(diffs) == 0:
            continue

        interval = diffs.mode().iloc[0]

        minutes = interval.total_seconds() / 60

        candidates.append(
            {
                "name": name,
                "df": obj,
                "minutes": minutes,
                "rows": len(obj),
            }
        )

    if not candidates:
        raise ValueError(
            "OHLC DataFrameがNotebook内に見つかりませんでした。"
        )

    print("=== 見つかったOHLC DataFrame ===")

    for c in candidates:
        print(
            f"{c['name']}: "
            f"{c['rows']:,} rows / "
            f"約{c['minutes']:.0f}分足"
        )

    # --------------------------------------------------------
    # 15分足を最優先
    # --------------------------------------------------------

    fifteen = [
        c for c in candidates
        if abs(c["minutes"] - 15) < 0.1
    ]

    if fifteen:
        chosen = max(
            fifteen,
            key=lambda x: x["rows"]
        )

        print(
            f"\n→ 15分足DataFrame "
            f"'{chosen['name']}' を使用します。"
        )

        return chosen["df"].copy()

    # --------------------------------------------------------
    # 15分足がなければ5分足
    # --------------------------------------------------------

    five = [
        c for c in candidates
        if abs(c["minutes"] - 5) < 0.1
    ]

    if five:
        chosen = max(
            five,
            key=lambda x: x["rows"]
        )

        print(
            f"\n→ 15分足がないため、"
            f"5分足 '{chosen['name']}' を使用します。"
        )

        return chosen["df"].copy()

    raise ValueError(
        "15分足または5分足のOHLC DataFrameが見つかりませんでした。"
    )


# ============================================================
# 2. 列名を統一
# ============================================================

def normalize_ohlc(df):

    x = df.copy()

    rename = {}

    for c in x.columns:

        name = str(c).lower()

        if name == "open":
            rename[c] = "open"

        elif name == "high":
            rename[c] = "high"

        elif name == "low":
            rename[c] = "low"

        elif name == "close":
            rename[c] = "close"

    x = x.rename(columns=rename)

    x = x[
        ["open", "high", "low", "close"]
    ].copy()

    x = x.apply(
        pd.to_numeric,
        errors="coerce"
    )

    x = x.dropna()

    x = x.sort_index()

    # 重複時刻を除去
    x = x[
        ~x.index.duplicated(
            keep="first"
        )
    ]

    return x


# ============================================================
# 3. 5分足なら15分足へ変換
# ============================================================

def convert_to_15min(df):

    x = normalize_ohlc(df)

    diffs = (
        x.index
        .to_series()
        .diff()
        .dropna()
    )

    interval = diffs.mode().iloc[0]

    minutes = (
        interval.total_seconds()
        / 60
    )

    print(
        f"\n元データの典型的間隔: "
        f"{minutes:.0f}分"
    )

    # --------------------------------------------------------
    # 既に15分足
    # --------------------------------------------------------

    if abs(minutes - 15) < 0.1:

        print(
            "→ 既に15分足なので"
            "そのまま使用します。"
        )

        return x

    # --------------------------------------------------------
    # 5分足 → 15分足
    # --------------------------------------------------------

    if abs(minutes - 5) < 0.1:

        print(
            "→ 5分足を15分足へ変換します。"
        )

        x15 = x.resample(
            "15min",
            label="left",
            closed="left"
        ).agg(
            {
                "open": "first",
                "high": "max",
                "low": "min",
                "close": "last",
            }
        )

        # 3本揃っていない15分足を除外するため
        count = (
            x["close"]
            .resample(
                "15min",
                label="left",
                closed="left"
            )
            .count()
        )

        x15 = x15.loc[
            count == 3
        ].dropna()

        print(
            f"変換後: {len(x15):,} rows"
        )

        return x15

    raise ValueError(
        f"{minutes:.1f}分足でした。"
        "現在は5分足または15分足だけ対応します。"
    )


# ============================================================
# 4. データを準備
# ============================================================

raw_bars = find_ohlc_dataframe()

bars15 = convert_to_15min(
    raw_bars
)


print("\n" + "=" * 60)

print("使用する15分足")

print("=" * 60)

print(
    f"行数: {len(bars15):,}"
)

print(
    f"開始: {bars15.index.min()}"
)

print(
    f"終了: {bars15.index.max()}"
)

print(
    bars15.head()
)


## 元セルindex 53


In [ ]:
# ============================================================
# CELL 1
# HGB Champion Pipeline
# データ + 30特徴量 + 30分先ラベル
# ============================================================

import math
import numpy as np
import pandas as pd

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import roc_auc_score, brier_score_loss


# -----------------------------
# 固定設定
# -----------------------------

FEATURES = [
    "return_1", "return_2", "return_4", "return_8", "return_16",
    "vol_4", "vol_8", "vol_16", "vol_32",
    "ma5_distance", "ma5_slope",
    "ma10_distance", "ma10_slope",
    "ma20_distance", "ma20_slope",
    "ma50_distance", "ma50_slope",
    "ma100_distance", "ma100_slope",
    "body", "upper_wick", "lower_wick", "range_pct",
    "rsi14", "atr14",
    "distance_high_16", "distance_low_16",
    "hour_sin", "hour_cos", "weekday",
]

THRESHOLDS = [
    0.50, 0.52, 0.54, 0.55, 0.56,
    0.58, 0.60, 0.62, 0.65
]

SESSIONS = [
    "ALL",
    "UTC_13_24",
    "UTC_21_24",
    "EXCLUDE_08_13",
]

SIZING_POLICIES = [
    "FIXED",
    "GENTLE",
    "MODERATE",
    "STRONG",
]

CALIBRATIONS = [
    "RAW",
    "PLATT",
    "ISOTONIC",
]

COST = 0.00004
MIN_VALIDATION_TRADES = 100

HGB_CONFIG = {
    "learning_rate": 0.05,
    "max_iter": 250,
    "max_leaf_nodes": 15,
    "min_samples_leaf": 30,
    "l2_regularization": 1.0,
    "early_stopping": False,
    "random_state": 42,
}


# ============================================================
# bars の確認
# ============================================================

if "bars" not in globals():
    raise NameError(
        "bars が見つかりません。"
        "さっき確認した15分足DataFrameを bars という名前で用意してください。"
    )

bars15 = bars.copy()

# OHLC列名を小文字へ
bars15.columns = [
    str(c).strip().lower()
    for c in bars15.columns
]

required_ohlc = ["open", "high", "low", "close"]

missing = [
    c for c in required_ohlc
    if c not in bars15.columns
]

if missing:
    raise ValueError(
        f"OHLC列が不足しています: {missing}"
    )

bars15 = bars15[required_ohlc].copy()


# DatetimeIndex確認
if not isinstance(bars15.index, pd.DatetimeIndex):
    raise TypeError(
        "bars.index が DatetimeIndex ではありません。"
    )


# UTCへ統一
if bars15.index.tz is None:
    bars15.index = bars15.index.tz_localize("UTC")
else:
    bars15.index = bars15.index.tz_convert("UTC")


bars15 = (
    bars15
    .sort_index()
    .loc[lambda x: ~x.index.duplicated(keep="first")]
)

bars15 = (
    bars15
    .apply(pd.to_numeric, errors="coerce")
    .dropna()
)


# ============================================================
# RSI
# ============================================================

def calc_rsi(close, period=14):

    delta = close.diff()

    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    avg_gain = gain.rolling(period).mean()
    avg_loss = loss.rolling(period).mean()

    rs = avg_gain / avg_loss.replace(0, np.nan)

    return 100 - 100 / (1 + rs)


# ============================================================
# 特徴量
# ============================================================

def make_features(df):

    x = df.copy()

    # Return
    for n in [1, 2, 4, 8, 16]:
        x[f"return_{n}"] = x["close"].pct_change(n)

    # Volatility
    for n in [4, 8, 16, 32]:
        x[f"vol_{n}"] = (
            x["return_1"]
            .rolling(n)
            .std()
        )

    # Moving Average
    for n in [5, 10, 20, 50, 100]:

        ma = x["close"].rolling(n).mean()

        x[f"ma{n}_distance"] = (
            x["close"] / ma - 1
        )

        x[f"ma{n}_slope"] = (
            ma.pct_change()
        )

    # Candle
    candle_range = (
        x["high"] - x["low"]
    ).replace(0, np.nan)

    x["body"] = (
        (x["close"] - x["open"])
        / candle_range
    )

    x["upper_wick"] = (
        x["high"]
        - x[["open", "close"]].max(axis=1)
    ) / candle_range

    x["lower_wick"] = (
        x[["open", "close"]].min(axis=1)
        - x["low"]
    ) / candle_range

    x["range_pct"] = (
        (x["high"] - x["low"])
        / x["close"]
    )

    # RSI
    x["rsi14"] = (
        calc_rsi(x["close"], 14)
        / 100
    )

    # ATR
    prev_close = x["close"].shift(1)

    tr = pd.concat(
        [
            x["high"] - x["low"],
            (x["high"] - prev_close).abs(),
            (x["low"] - prev_close).abs(),
        ],
        axis=1,
    ).max(axis=1)

    x["atr14"] = (
        tr.rolling(14).mean()
        / x["close"]
    )

    # 16本 High / Low
    high16 = x["high"].rolling(16).max()
    low16 = x["low"].rolling(16).min()

    x["distance_high_16"] = (
        (high16 - x["close"])
        / x["close"]
    )

    x["distance_low_16"] = (
        (x["close"] - low16)
        / x["close"]
    )

    # 時刻
    hour = (
        x.index.hour
        + x.index.minute / 60
    )

    x["hour_sin"] = np.sin(
        2 * np.pi * hour / 24
    )

    x["hour_cos"] = np.cos(
        2 * np.pi * hour / 24
    )

    x["weekday"] = (
        x.index.dayofweek / 4
    )

    return x.replace(
        [np.inf, -np.inf],
        np.nan
    )


# ============================================================
# 30分固定 Exit
#
# signal = t
# entry  = Open(t+1)
# exit   = Close(t+2)
# ============================================================

data_hgb = make_features(bars15)

times = pd.Series(
    bars15.index,
    index=bars15.index
)

data_hgb["entry_time"] = (
    times.shift(-1)
)

data_hgb["label_end"] = (
    times.shift(-2)
    + pd.Timedelta(minutes=15)
)

data_hgb["entry_price"] = (
    bars15["open"].shift(-1)
)

data_hgb["exit_price"] = (
    bars15["close"].shift(-2)
)

data_hgb["future_return"] = (
    data_hgb["exit_price"]
    / data_hgb["entry_price"]
    - 1
)

data_hgb["target"] = (
    data_hgb["future_return"] > 0
).astype(int)


# t → t+1 → t+2 が連続15分足かチェック
continuous = (
    (times.shift(-1) - times)
    .eq(pd.Timedelta(minutes=15))
    &
    (times.shift(-2) - times)
    .eq(pd.Timedelta(minutes=30))
)

need = FEATURES + [
    "entry_time",
    "label_end",
    "entry_price",
    "exit_price",
    "future_return",
]

data_hgb = (
    data_hgb.loc[continuous]
    .dropna(subset=need)
    .copy()
)


print("=" * 70)
print("DATASET READY")
print("=" * 70)

print("Bars:", len(bars15))
print("ML rows:", len(data_hgb))
print("Features:", len(FEATURES))

print(
    "Period:",
    data_hgb.index.min(),
    "→",
    data_hgb.index.max(),
)

print(
    "Target UP rate:",
    data_hgb["target"].mean(),
)

print("\nCELL 1 完了")

# ============================================================
# CELL 2
# HGB + Annual Nested Walk-Forward + Calibration
# ============================================================


def make_hgb():

    return HistGradientBoostingClassifier(
        **HGB_CONFIG
    )


def fit_hgb(train):

    if len(train) < 5000:
        raise ValueError(
            "学習データが5000行未満です。"
        )

    if train["target"].nunique() < 2:
        raise ValueError(
            "targetが1クラスしかありません。"
        )

    model = make_hgb()

    model.fit(
        train[FEATURES],
        train["target"]
    )

    return model


def predict_probability(model, frame):

    return model.predict_proba(
        frame[FEATURES]
    )[:, 1]


# ============================================================
# Annual split
# ============================================================

def annual_split(data, test_year):

    validation_year = test_year - 1

    val_start = pd.Timestamp(
        f"{validation_year}-01-01",
        tz="UTC"
    )

    test_start = pd.Timestamp(
        f"{test_year}-01-01",
        tz="UTC"
    )

    test_end = pd.Timestamp(
        f"{test_year + 1}-01-01",
        tz="UTC"
    )

    train = data.loc[
        (data.index < val_start)
        &
        (data["label_end"] <= val_start)
    ].copy()

    validation = data.loc[
        (data.index >= val_start)
        &
        (data.index < test_start)
        &
        (data["label_end"] <= test_start)
    ].copy()

    final_train = data.loc[
        (data.index < test_start)
        &
        (data["label_end"] <= test_start)
    ].copy()

    test = data.loc[
        (data.index >= test_start)
        &
        (data.index < test_end)
        &
        (data["label_end"] <= test_end)
    ].copy()

    if (
        len(train) < 5000
        or len(validation) < 100
        or len(test) < 100
    ):
        return None

    return {
        "test_year": test_year,
        "validation_year": validation_year,
        "train": train,
        "validation": validation,
        "final_train": final_train,
        "test": test,
    }


# ============================================================
# Forward OOF
# ============================================================

def make_oof(train_history):

    pieces = []

    years = sorted(
        train_history.index.year.unique()
    )

    for year in years:

        previous_years = [
            y for y in years
            if y < year
        ]

        if len(previous_years) < 2:
            continue

        start = pd.Timestamp(
            f"{year}-01-01",
            tz="UTC"
        )

        end = pd.Timestamp(
            f"{year + 1}-01-01",
            tz="UTC"
        )

        hist = train_history.loc[
            (train_history.index < start)
            &
            (train_history["label_end"] <= start)
        ]

        oof = train_history.loc[
            (train_history.index >= start)
            &
            (train_history.index < end)
            &
            (train_history["label_end"] <= end)
        ]

        if (
            len(hist) < 5000
            or len(oof) < 100
        ):
            continue

        model = fit_hgb(hist)

        p = predict_probability(
            model,
            oof
        )

        piece = pd.DataFrame(
            {
                "probability": p,
                "target": oof["target"].values,
            },
            index=oof.index
        )

        pieces.append(piece)

    if not pieces:

        return pd.DataFrame(
            columns=[
                "probability",
                "target"
            ]
        )

    return pd.concat(
        pieces
    ).sort_index()


# ============================================================
# Calibration
# ============================================================

class RawCalibration:

    def fit(self, p, y):
        return self

    def predict(self, p):
        return np.asarray(p)


class PlattCalibration:

    def __init__(self):

        self.model = LogisticRegression(
            solver="lbfgs",
            random_state=42
        )

    def fit(self, p, y):

        self.model.fit(
            np.asarray(p).reshape(-1, 1),
            y
        )

        return self

    def predict(self, p):

        return self.model.predict_proba(
            np.asarray(p).reshape(-1, 1)
        )[:, 1]


class IsotonicCalibration:

    def __init__(self):

        self.model = IsotonicRegression(
            y_min=0,
            y_max=1,
            out_of_bounds="clip"
        )

    def fit(self, p, y):

        self.model.fit(p, y)

        return self

    def predict(self, p):

        return self.model.predict(p)


def fit_calibrator(method, oof):

    if method == "RAW":
        return RawCalibration()

    # OOF不足なら安全にRAWへ戻す
    if (
        len(oof) < 500
        or oof["target"].nunique() < 2
    ):
        return RawCalibration()

    p = oof["probability"].values
    y = oof["target"].values

    if method == "PLATT":

        return (
            PlattCalibration()
            .fit(p, y)
        )

    if method == "ISOTONIC":

        if len(oof) < 1000:
            return RawCalibration()

        return (
            IsotonicCalibration()
            .fit(p, y)
        )

    raise ValueError(method)


def ece_score(y, p, bins=10):

    y = np.asarray(y)
    p = np.asarray(p)

    edges = np.linspace(
        0, 1, bins + 1
    )

    ids = np.digitize(
        p,
        edges[1:-1]
    )

    ece = 0

    for b in range(bins):

        mask = ids == b

        if not mask.any():
            continue

        ece += (
            mask.mean()
            *
            abs(
                y[mask].mean()
                -
                p[mask].mean()
            )
        )

    return float(ece)


def choose_calibration(
    train,
    validation
):

    oof = make_oof(train)

    model = fit_hgb(train)

    raw_val = predict_probability(
        model,
        validation
    )

    rows = []

    for method in CALIBRATIONS:

        calibrator = fit_calibrator(
            method,
            oof
        )

        p = np.clip(
            calibrator.predict(raw_val),
            0,
            1
        )

        brier = brier_score_loss(
            validation["target"],
            p
        )

        ece = ece_score(
            validation["target"].values,
            p
        )

        auc = roc_auc_score(
            validation["target"],
            p
        )

        rows.append(
            {
                "method": method,
                "brier": brier,
                "ece": ece,
                "auc": auc,
            }
        )

    table = pd.DataFrame(rows)

    table = table.sort_values(
        [
            "brier",
            "ece",
            "method"
        ]
    )

    chosen = table.iloc[0]["method"]

    return (
        chosen,
        table,
        raw_val
    )


print("CELL 2 完了")

# ============================================================
# CELL 3
# Threshold + Session + Position Sizing
# ============================================================


def session_mask(index, policy):

    hour = index.hour

    if policy == "ALL":
        return np.ones(
            len(index),
            dtype=bool
        )

    if policy == "UTC_13_24":
        return (
            (hour >= 13)
            &
            (hour < 24)
        )

    if policy == "UTC_21_24":
        return (
            (hour >= 21)
            &
            (hour < 24)
        )

    if policy == "EXCLUDE_08_13":
        return ~(
            (hour >= 8)
            &
            (hour < 13)
        )

    raise ValueError(policy)


def make_predictions(
    frame,
    probability
):

    p = np.asarray(probability)

    sign = np.where(
        p >= 0.5,
        1,
        -1
    )

    out = frame[
        [
            "entry_time",
            "label_end",
            "future_return",
            "target"
        ]
    ].copy()

    out["p_up"] = p

    out["confidence"] = np.maximum(
        p,
        1 - p
    )

    out["direction"] = np.where(
        p >= 0.5,
        "BUY",
        "SELL"
    )

    out["gross_return"] = (
        frame["future_return"].values
        *
        sign
    )

    return out


def select_trades(
    predictions,
    threshold,
    session
):

    mask = (
        (
            predictions["confidence"]
            >= threshold
        )
        &
        session_mask(
            predictions.index,
            session
        )
    )

    candidates = (
        predictions
        .loc[mask]
        .sort_index()
    )

    selected = []

    next_free_time = None

    for row in candidates.itertuples():

        if (
            next_free_time is not None
            and
            row.entry_time < next_free_time
        ):
            continue

        selected.append(
            row.Index
        )

        next_free_time = (
            row.label_end
        )

    trades = candidates.loc[
        selected
    ].copy()

    trades["net_return"] = (
        trades["gross_return"]
        -
        COST
    )

    return trades


def stats(return_series):

    r = np.asarray(
        return_series,
        dtype=float
    )

    r = r[
        np.isfinite(r)
    ]

    if len(r) == 0:

        return {
            "trades": 0,
            "win_rate": np.nan,
            "avg_return": np.nan,
            "profit_factor": np.nan,
            "growth": 0,
            "max_dd": np.nan,
            "return_to_dd": np.nan,
        }

    gain = r[r > 0].sum()

    loss = -r[r < 0].sum()

    pf = (
        gain / loss
        if loss > 0
        else np.inf
    )

    equity = np.r_[
        1,
        np.cumprod(1 + r)
    ]

    peak = np.maximum.accumulate(
        equity
    )

    dd = equity / peak - 1

    max_dd = dd.min()

    growth = (
        equity[-1] - 1
    )

    return_dd = (
        growth / abs(max_dd)
        if max_dd < 0
        else np.nan
    )

    return {
        "trades": len(r),
        "win_rate": (r > 0).mean(),
        "avg_return": r.mean(),
        "profit_factor": pf,
        "growth": growth,
        "max_dd": max_dd,
        "return_to_dd": return_dd,
    }


# ============================================================
# Threshold + Session
# ============================================================

def choose_threshold_session(
    validation_predictions
):

    rows = []

    best = None
    best_key = None

    for threshold in THRESHOLDS:

        for session in SESSIONS:

            trades = select_trades(
                validation_predictions,
                threshold,
                session
            )

            s = stats(
                trades["net_return"]
            )

            eligible = (
                s["trades"]
                >= MIN_VALIDATION_TRADES
            )

            score = (
                s["avg_return"]
                *
                np.sqrt(s["trades"])
                if eligible
                else -np.inf
            )

            rows.append(
                {
                    "threshold": threshold,
                    "session": session,
                    "score": score,
                    **s
                }
            )

            if not eligible:
                continue

            key = (
                score,
                s["profit_factor"],
                s["trades"],
                -threshold
            )

            if (
                best_key is None
                or key > best_key
            ):

                best_key = key

                best = (
                    threshold,
                    session
                )

    if best is None:

        raise RuntimeError(
            "Validationで100取引以上の"
            "Threshold/Session候補がありません。"
        )

    return (
        best[0],
        best[1],
        pd.DataFrame(rows)
    )


# ============================================================
# Position Sizing
# ============================================================

def raw_size(
    confidence,
    threshold,
    policy
):

    edge = np.clip(
        (
            confidence
            - threshold
        )
        /
        max(
            1 - threshold,
            1e-8
        ),
        0,
        1
    )

    if policy == "FIXED":

        return np.ones_like(edge)

    if policy == "GENTLE":

        return (
            0.85
            +
            0.30 * edge
        )

    if policy == "MODERATE":

        return (
            0.70
            +
            0.60 * edge
        )

    if policy == "STRONG":

        return (
            0.50
            +
            1.00 * edge
        )

    raise ValueError(policy)


def choose_sizing(
    trades,
    threshold
):

    rows = []

    for policy in SIZING_POLICIES:

        raw = raw_size(
            trades["confidence"].values,
            threshold,
            policy
        )

        # Validation平均Exposure=1
        scale = (
            1 / raw.mean()
        )

        size = raw * scale

        returns = (
            size
            *
            trades["net_return"].values
        )

        s = stats(returns)

        rows.append(
            {
                "policy": policy,
                "scale": scale,
                **s
            }
        )

    table = pd.DataFrame(rows)

    fixed = table.loc[
        table["policy"] == "FIXED"
    ].iloc[0]

    valid = []

    for _, row in table.iterrows():

        if row["policy"] == "FIXED":
            continue

        if (
            row["avg_return"]
            >= fixed["avg_return"]
            and
            row["profit_factor"]
            >= fixed["profit_factor"]
            and
            row["return_to_dd"]
            >= fixed["return_to_dd"]
        ):

            valid.append(row)

    if not valid:

        chosen = fixed

    else:

        chosen = max(
            valid,
            key=lambda x: (
                x["return_to_dd"],
                x["profit_factor"],
                x["avg_return"]
            )
        )

    return (
        chosen["policy"],
        float(chosen["scale"]),
        table
    )


def apply_sizing(
    trades,
    threshold,
    policy,
    scale,
    cost_multiplier=1
):

    out = trades.copy()

    if len(out) == 0:

        out["position_size"] = []
        out["sized_return"] = []

        return out

    size = raw_size(
        out["confidence"].values,
        threshold,
        policy
    )

    size = size * scale

    # 安全Cap
    size = np.clip(
        size,
        0.25,
        2.0
    )

    out["position_size"] = size

    out["sized_return"] = (
        size
        *
        (
            out["gross_return"]
            -
            COST * cost_multiplier
        )
    )

    return out


print("CELL 3 完了")

# ============================================================
# CELL 4
# Complete Nested Walk-Forward
# 2020-2026
# ============================================================


def evaluate_year(test_year):

    split = annual_split(
        data_hgb,
        test_year
    )

    if split is None:

        print(
            f"{test_year}: SKIP"
        )

        return None, None

    train = split["train"]
    validation = split["validation"]
    final_train = split["final_train"]
    test = split["test"]


    # ---------------------------------
    # 1. CalibrationをValidationで選択
    # ---------------------------------

    calibration, cal_table, raw_val = (
        choose_calibration(
            train,
            validation
        )
    )

    oof_train = make_oof(train)

    calibrator = fit_calibrator(
        calibration,
        oof_train
    )

    p_val = np.clip(
        calibrator.predict(raw_val),
        0,
        1
    )

    val_predictions = make_predictions(
        validation,
        p_val
    )


    # ---------------------------------
    # 2. Threshold + Session
    # ---------------------------------

    threshold, session, threshold_table = (
        choose_threshold_session(
            val_predictions
        )
    )

    val_selected = select_trades(
        val_predictions,
        threshold,
        session
    )


    # ---------------------------------
    # 3. Position Sizing
    # ---------------------------------

    sizing_policy, sizing_scale, sizing_table = (
        choose_sizing(
            val_selected,
            threshold
        )
    )


    # ---------------------------------
    # 4. Test前に全履歴で再学習
    # ---------------------------------

    final_oof = make_oof(
        final_train
    )

    final_calibrator = fit_calibrator(
        calibration,
        final_oof
    )

    model = fit_hgb(
        final_train
    )

    raw_test = predict_probability(
        model,
        test
    )

    p_test = np.clip(
        final_calibrator.predict(
            raw_test
        ),
        0,
        1
    )

    test_predictions = make_predictions(
        test,
        p_test
    )

    selected = select_trades(
        test_predictions,
        threshold,
        session
    )

    trades = apply_sizing(
        selected,
        threshold,
        sizing_policy,
        sizing_scale
    )

    s = stats(
        trades["sized_return"]
    )

    auc = roc_auc_score(
        test["target"],
        p_test
    )

    result = {
        "test_year": test_year,
        "validation_year": test_year - 1,
        "calibration": calibration,
        "threshold": threshold,
        "session": session,
        "sizing_policy": sizing_policy,
        "auc": auc,
        **s
    }

    trades = trades.copy()

    trades["test_year"] = (
        test_year
    )

    return result, trades


# ============================================================
# 全年実行
# ============================================================

annual_results = []
trade_results = []

for year in range(
    2020,
    2027
):

    print("\n")
    print("=" * 70)
    print(
        "TEST YEAR:",
        year
    )
    print("=" * 70)

    try:

        result, trades = (
            evaluate_year(year)
        )

        if result is None:
            continue

        annual_results.append(
            result
        )

        trade_results.append(
            trades
        )

        print(
            "Calibration:",
            result["calibration"]
        )

        print(
            "Threshold:",
            result["threshold"]
        )

        print(
            "Session:",
            result["session"]
        )

        print(
            "Sizing:",
            result["sizing_policy"]
        )

        print(
            "AUC:",
            round(result["auc"], 4)
        )

        print(
            "Trades:",
            result["trades"]
        )

        print(
            "Win:",
            round(
                result["win_rate"] * 100,
                2
            ),
            "%"
        )

        print(
            "Avg Return:",
            round(
                result["avg_return"] * 100,
                5
            ),
            "%"
        )

        print(
            "PF:",
            round(
                result["profit_factor"],
                3
            )
        )

        print(
            "Growth:",
            round(
                result["growth"] * 100,
                3
            ),
            "%"
        )

        print(
            "Max DD:",
            round(
                result["max_dd"] * 100,
                3
            ),
            "%"
        )

        print(
            "Return/DD:",
            round(
                result["return_to_dd"],
                3
            )
        )

    except Exception as e:

        print(
            "ERROR:",
            type(e).__name__,
            e
        )


annual_hgb = pd.DataFrame(
    annual_results
)

all_hgb_trades = (
    pd.concat(
        trade_results
    ).sort_index()
    if trade_results
    else pd.DataFrame()
)


print("\n")
print("=" * 80)
print("ANNUAL HGB RESULTS")
print("=" * 80)

display(
    annual_hgb
)


# ============================================================
# Development OOS
# ============================================================

development = all_hgb_trades.loc[
    all_hgb_trades["test_year"]
    .between(2020, 2025)
]

dev_stats = stats(
    development["sized_return"]
)


print("\n")
print("=" * 80)
print("DEVELOPMENT OOS 2020-2025")
print("=" * 80)

print(
    "Trades:",
    dev_stats["trades"]
)

print(
    "Win:",
    dev_stats["win_rate"] * 100,
    "%"
)

print(
    "Avg Return:",
    dev_stats["avg_return"] * 100,
    "%"
)

print(
    "PF:",
    dev_stats["profit_factor"]
)

print(
    "Growth:",
    dev_stats["growth"] * 100,
    "%"
)

print(
    "Max DD:",
    dev_stats["max_dd"] * 100,
    "%"
)

print(
    "Return/DD:",
    dev_stats["return_to_dd"]
)


# ============================================================
# 2026 Confirmation
# ============================================================

confirmation = all_hgb_trades.loc[
    all_hgb_trades["test_year"]
    == 2026
]

confirm_stats = stats(
    confirmation["sized_return"]
)


print("\n")
print("=" * 80)
print("CONFIRMATION 2026")
print("=" * 80)

print(
    confirm_stats
)


# ============================================================
# 選択安定性
# ============================================================

print("\n")
print("=" * 80)
print("SELECTION FREQUENCY")
print("=" * 80)

print("\nCalibration")
print(
    annual_hgb[
        "calibration"
    ].value_counts()
)

print("\nThreshold")
print(
    annual_hgb[
        "threshold"
    ].value_counts().sort_index()
)

print("\nSession")
print(
    annual_hgb[
        "session"
    ].value_counts()
)

print("\nSizing")
print(
    annual_hgb[
        "sizing_policy"
    ].value_counts()
)


# ============================================================
# Cost Stress
# ============================================================

print("\n")
print("=" * 80)
print("COST STRESS")
print("=" * 80)

cost_rows = []

for multiplier in [
    1,
    1.5,
    2
]:

    r = (
        development[
            "position_size"
        ].values
        *
        (
            development[
                "gross_return"
            ].values
            -
            COST * multiplier
        )
    )

    s = stats(r)

    cost_rows.append(
        {
            "cost_x": multiplier,
            "cost_pct":
                COST
                * multiplier
                * 100,
            **s
        }
    )

cost_stress_hgb = pd.DataFrame(
    cost_rows
)

display(
    cost_stress_hgb
)


# ============================================================
# 最終診断
# ============================================================

dev_years = annual_hgb.loc[
    annual_hgb[
        "test_year"
    ].between(2020, 2025)
]

positive_years = (
    dev_years["avg_return"] > 0
).sum()

pf_years = (
    dev_years["profit_factor"] > 1
).sum()


print("\n")
print("=" * 80)
print("FINAL DIAGNOSIS")
print("=" * 80)

print(
    "Development positive years:",
    positive_years,
    "/",
    len(dev_years)
)

print(
    "Development PF > 1 years:",
    pf_years,
    "/",
    len(dev_years)
)

confirmation_ok = (
    confirm_stats["avg_return"] > 0
    and
    confirm_stats["profit_factor"] > 1
)

cost2 = cost_stress_hgb.loc[
    cost_stress_hgb["cost_x"] == 2
].iloc[0]

cost2_ok = (
    cost2["avg_return"] > 0
    and
    cost2["profit_factor"] > 1
)

robust = (
    positive_years >= 5
    and
    pf_years >= 5
    and
    dev_stats["avg_return"] > 0
    and
    dev_stats["profit_factor"] > 1
)


if (
    robust
    and confirmation_ok
    and cost2_ok
):

    FINAL_DECISION = (
        "HGB FULL PIPELINE ROBUST"
    )

elif (
    robust
    and confirmation_ok
):

    FINAL_DECISION = (
        "HGB ROBUST / COST要確認"
    )

elif robust:

    FINAL_DECISION = (
        "HGB DEVELOPMENT ROBUST / "
        "2026要確認"
    )

else:

    FINAL_DECISION = (
        "HGB NOT ROBUST ENOUGH"
    )


print("\n")
print(
    "FINAL DECISION:",
    FINAL_DECISION
)

print("\n検証終了")
